# 🏴‍☠️ OpenViking en Google Colab (Cloud Edition)

Este notebook permite correr el agente **OpenViking** con memoria persistente en la nube usando **Firebase Firestore**.

### 1. Configuración de Secretos
Ve al icono de la llave (🔑) a la izquierda y añade:
- **`TELEGRAM_TOKEN`**: Tu token de BotFather.
- **`NGROK_AUTHTOKEN`**: Tu token de ngrok (para SSH).
- **`FIREBASE_PROJECT_ID`**: El ID del proyecto que crearemos abajo.

In [ ]:
# @title ⚙️ Configurar Entorno y Clonar
import os
repo_url = "https://github.com/codigo8a/OpenViking-Python.git"
branch = "google-colab"
repo_name = "OpenViking-Python"

if not os.path.exists('agent.py'):
    print("📥 Descargando OpenViking...")
    !git clone -b {branch} {repo_url}
    %cd {repo_name}
else:
    print("✅ Proyecto ya configurado.")

!pip install -q requests python-telegram-bot firebase-admin python-dotenv pyngrok mcp

In [ ]:
# @title 🔥 Crear Proyecto Firebase (Elegir un ID único)
import random
import string

suggested_id = "openviking-cloud-" + "".join(random.choices(string.digits, k=4))
PROJECT_ID = suggested_id # @param {type:"string"}

print(f"🚀 Creando proyecto Firebase: {PROJECT_ID}...")
!firebase projects:create {PROJECT_ID} --display-name "OpenViking Memory"

print(f"\n✅ ¡PROYECTO CREADO! Copia este ID y ponlo en tus Secretos (🔑) como FIREBASE_PROJECT_ID")
print(f"👉 ID: {PROJECT_ID}")

print("\n⚠️ IMPORTANTE: Ve a la consola de Firebase y habilita FIRESTORE en este proyecto para que la memoria funcione.")

In [ ]:
# @title 🚀 Levantar Ollama (Local LLM)
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

print('⏳ Iniciando Ollama...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15)
!ollama pull llama3:8b
print('✅ Ollama listo con Llama3:8b')

In [ ]:
# @title 🤖 Iniciar Agente (Modo CLI)
from agent import OpenVikingAgent
from google.colab import userdata
import os

KEYS = ['GOOGLE_API_KEY', 'GOOGLE_SEARCH_ENGINE_ID', 'GROQ_API_KEY', 'TELEGRAM_TOKEN', 'NGROK_AUTHTOKEN', 'FIREBASE_PROJECT_ID']
for k in KEYS:
    try: os.environ[k] = userdata.get(k)
    except: pass

agent = OpenVikingAgent()
print("✅ OpenViking listo.")
task = "¿Qué tal la memoria en Firebase?" # @param {type:"string"}
print(agent.execute_task(task))

In [ ]:
# @title 🚀 Iniciar Bot de Telegram
from google.colab import userdata
import os

KEYS = ['TELEGRAM_TOKEN', 'NGROK_AUTHTOKEN', 'FIREBASE_PROJECT_ID', 'GOOGLE_API_KEY']
env_str = ""
for k in KEYS:
    try:
        val = userdata.get(k)
        if val:
            os.environ[k] = val
            env_str += f'{k}="{val}" '
    except: pass

if os.environ.get('TELEGRAM_TOKEN'):
    print("📡 Iniciando servidor de Telegram con Firebase Cloud Memory...")
    !{env_str} python telegram_bot.py
else:
    print("❌ ERROR: No se encontró TELEGRAM_TOKEN.")